In [2]:
import numpy as np
import pandas as pd

medical_df=pd.read_csv('/kaggle/input/datasets/fedesoriano/heart-failure-prediction/heart.csv')
print(medical_df.info)
print(medical_df.describe())

<bound method DataFrame.info of      Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  \
0     40   M           ATA        140          289          0     Normal   
1     49   F           NAP        160          180          0     Normal   
2     37   M           ATA        130          283          0         ST   
3     48   F           ASY        138          214          0     Normal   
4     54   M           NAP        150          195          0     Normal   
..   ...  ..           ...        ...          ...        ...        ...   
913   45   M            TA        110          264          0     Normal   
914   68   M           ASY        144          193          1     Normal   
915   57   M           ASY        130          131          0     Normal   
916   57   F           ATA        130          236          0        LVH   
917   38   M           NAP        138          175          0     Normal   

     MaxHR ExerciseAngina  Oldpeak ST_Slope  HeartDisea

In [20]:
print(medical_df.isna().sum())
print(medical_df.duplicated().sum())

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64
0


In [21]:
print((medical_df == 0).sum())

Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS         704
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak           368
ST_Slope            0
HeartDisease      410
dtype: int64


In [22]:
print(medical_df['HeartDisease'].value_counts())

HeartDisease
1    508
0    410
Name: count, dtype: int64


In [23]:
print(medical_df.dtypes)

Age                 int64
Sex                object
ChestPainType      object
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG         object
MaxHR               int64
ExerciseAngina     object
Oldpeak           float64
ST_Slope           object
HeartDisease        int64
dtype: object


In [3]:
median_chol = medical_df[medical_df['Cholesterol'] != 0]['Cholesterol'].median()
medical_df['Cholesterol'] = medical_df['Cholesterol'].replace(0, median_chol)

median_bp = medical_df[medical_df['RestingBP'] != 0]['RestingBP'].median()
medical_df['RestingBP'] = medical_df['RestingBP'].replace(0, median_bp)

In [5]:
from sklearn.preprocessing import LabelEncoder

le_sex      = LabelEncoder()
le_exercise = LabelEncoder()

medical_df['Sex'] = le_sex.fit_transform(medical_df['Sex'])
medical_df['ExerciseAngina'] = le_exercise.fit_transform(medical_df['ExerciseAngina'])

print(medical_df['Sex'].unique())       
print(medical_df['ExerciseAngina'].unique())

[1 0]
[0 1]


In [6]:
import pickle
with open('le_sex.pkl', 'wb') as f:
    pickle.dump(le_sex, f)

with open('le_exercise.pkl', 'wb') as f:
    pickle.dump(le_exercise, f)

In [26]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

cat_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
ohe = OneHotEncoder(sparse_output=False)

encoded_array = ohe.fit_transform(medical_df[cat_cols])

encoded_df = pd.DataFrame(encoded_array, columns=ohe.get_feature_names_out(cat_cols))

medical_df = medical_df.drop(columns=cat_cols)

medical_df = pd.concat([medical_df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

print(medical_df.head())
print(medical_df.shape)


   Age  Sex  RestingBP  Cholesterol  FastingBS  MaxHR  ExerciseAngina  \
0   40    1        140          289          0    172               0   
1   49    0        160          180          0    156               0   
2   37    1        130          283          0     98               0   
3   48    0        138          214          0    108               1   
4   54    1        150          195          0    122               0   

   Oldpeak  HeartDisease  ChestPainType_ASY  ChestPainType_ATA  \
0      0.0             0                0.0                1.0   
1      1.0             1                0.0                0.0   
2      0.0             0                0.0                1.0   
3      1.5             1                1.0                0.0   
4      0.0             0                0.0                0.0   

   ChestPainType_NAP  ChestPainType_TA  RestingECG_LVH  RestingECG_Normal  \
0                0.0               0.0             0.0                1.0   
1         

In [27]:
print(medical_df.columns.tolist())

['Age', 'Sex', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'HeartDisease', 'ChestPainType_ASY', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST', 'ST_Slope_Down', 'ST_Slope_Flat', 'ST_Slope_Up']


In [28]:
from sklearn.preprocessing import StandardScaler
scale_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
scaler = StandardScaler()

medical_df[scale_cols] = scaler.fit_transform(medical_df[scale_cols])

print(medical_df[scale_cols].head())

        Age  RestingBP  Cholesterol     MaxHR   Oldpeak
0 -1.433140   0.415002     0.858035  1.382928 -0.832432
1 -0.478484   1.527329    -1.184227  0.754157  0.105664
2 -1.751359  -0.141161     0.745617 -1.525138 -0.832432
3 -0.584556   0.303769    -0.547191 -1.132156  0.574711
4  0.051881   0.971166    -0.903182 -0.581981 -0.832432


In [29]:
from sklearn.model_selection import train_test_split
import numpy as np

X = medical_df.drop(columns=['HeartDisease'])
y = medical_df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (734, 18)
X_test: (184, 18)
y_train: (734,)
y_test: (184,)


In [30]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

random_model = RandomForestClassifier(n_estimators=100, random_state=42)

random_model.fit(X_train, y_train)

y_pred = random_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.8695652173913043

Confusion Matrix:
 [[68  9]
 [15 92]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.88      0.85        77
           1       0.91      0.86      0.88       107

    accuracy                           0.87       184
   macro avg       0.87      0.87      0.87       184
weighted avg       0.87      0.87      0.87       184



In [31]:
import torch
from torch.utils.data import Dataset, DataLoader

class HeartDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [32]:
train_dataset = HeartDataset(X_train, y_train)
test_dataset  = HeartDataset(X_test, y_test)

print(len(train_dataset))
print(len(test_dataset))   

734
184


In [33]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("Total train batches:", len(train_loader))
print("Total test batches:", len(test_loader))

Total train batches: 23
Total test batches: 6


In [34]:
 print(X_train.shape)

(734, 18)


In [35]:
import torch
import torch.nn as nn

class HeartModelV1(nn.Module):
    def __init__(self):
        super().__init__()
        self.heart_mode = nn.Sequential(
            nn.Linear(in_features=18,out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64,out_features=32),
            nn.ReLU(),
            nn.Linear(in_features=32, out_features=1),
        )

    def forward(self, x):
        return self.heart_mode(x)

model_0 = HeartModelV1()
print(model_0)

HeartModelV1(
  (heart_mode): Sequential(
    (0): Linear(in_features=18, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [36]:
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

In [37]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() 
    acc = (correct / len(y_pred)) * 100 
    return acc

In [39]:
torch.manual_seed(42)
epochs = 100

for epoch in range(epochs):
    model_0.train()
    train_loss = 0
    train_acc = 0
    #forward
    for X_batch, y_batch in train_loader:
        y_logits = model_0(X_batch).squeeze()
        y_pred = torch.round(torch.sigmoid(y_logits))
        #loss
        loss = loss_fn(y_logits, y_batch)
        acc = accuracy_fn(y_true=y_batch, y_pred=y_pred)
        
        train_loss += loss.item()
        train_acc += acc
        #optimizer zero grad
        optimizer.zero_grad()
        #loss back
        loss.backward()
        #optimizer
        optimizer.step()

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    model_0.eval()
    test_loss = 0
    test_acc = 0

    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            test_logits = model_0(X_batch).squeeze()
            test_pred = torch.round(torch.sigmoid(test_logits))
            test_loss += loss_fn(test_logits, y_batch).item()
            test_acc += accuracy_fn(y_true=y_batch, y_pred=test_pred)

        test_loss /= len(test_loader)
        test_acc /= len(test_loader)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {train_loss:.5f}, Acc: {train_acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")

Epoch: 0 | Loss: 0.18368, Acc: 91.54% | Test Loss: 0.51366, Test Acc: 81.94%
Epoch: 10 | Loss: 0.15881, Acc: 93.84% | Test Loss: 0.41129, Test Acc: 87.67%
Epoch: 20 | Loss: 0.14146, Acc: 95.24% | Test Loss: 0.66067, Test Acc: 81.42%
Epoch: 30 | Loss: 0.12114, Acc: 95.50% | Test Loss: 0.59484, Test Acc: 83.68%
Epoch: 40 | Loss: 0.12932, Acc: 95.09% | Test Loss: 0.62235, Test Acc: 86.11%
Epoch: 50 | Loss: 0.09940, Acc: 97.00% | Test Loss: 0.62016, Test Acc: 86.11%
Epoch: 60 | Loss: 0.11152, Acc: 95.37% | Test Loss: 0.58087, Test Acc: 85.42%
Epoch: 70 | Loss: 0.11028, Acc: 95.37% | Test Loss: 0.62627, Test Acc: 86.46%
Epoch: 80 | Loss: 0.05842, Acc: 98.36% | Test Loss: 0.84673, Test Acc: 83.51%
Epoch: 90 | Loss: 0.04410, Acc: 99.30% | Test Loss: 0.73456, Test Acc: 83.33%


In [42]:
import pickle
import torch

# Save Random Forest
with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(random_model, f)
print("✅ Random Forest saved")

# Save Scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved")

# Save OneHotEncoder
with open('ohe.pkl', 'wb') as f:
    pickle.dump(ohe, f)
print("✅ OneHotEncoder saved")

# Save LabelEncoder
with open('le.pkl', 'wb') as f:
    pickle.dump(le, f)
print("✅ LabelEncoder saved")

# Save PyTorch model
torch.save(model_0.state_dict(), 'pytorch_model.pth')
print("✅ PyTorch model saved")

✅ Random Forest saved
✅ Scaler saved
✅ OneHotEncoder saved
✅ LabelEncoder saved
✅ PyTorch model saved
